In [ ]:
# 1.导入模块
import time           # 时间库       用来记录插入数据库时的当前时间
import numpy as np    # numpy库      用来做基本的数据处理
import pandas as pd   # pandas库     有关日期转换、数据格式化处理、主要RFM计算过程等  R：最近购买间隔，F：频次， M：金额
import pymysql        # mysql连接库   数据库连接工具，读写MySQL数据库
from sqlalchemy import create_engine   #导入引擎（可以使用MySQL）

!pip install pyecharts     #下载
from pyecharts.charts import Bar3D     # 绘制3d柱形图（此三行代码）
from pyecharts.commons.utils import JsCode
import pyecharts.options as opts

# 加载数据

In [ ]:
# 原始Excel表有5个sheet表，分别为'2015','2016','2017','2018','会员等级'
# 每张表内有4列：会员ID，订单号，提交日期，订单金额

# 1.1定义列表，记录；数据Excel表名
sheet_names = ['2015','2016','2017','2018','会员等级']
# 1.2从Excel中读取数据，获取到字典形式 → {'2015':df对象，'2016':df对象,......}
# 参1 ：Excel文件（路径） 参2 ：Excel文件中的表名
sheet_dict = pd.read_excel('./基本数据/2015~2018+会员等级表.xlsx',sheet_name=sheet_names)
sheet_dict

In [ ]:
# 1.3查看sheet_dict的变量数据类型
type(sheet_dict)      # class 'dict' 字典类型

# 1.4查看2015 excel表中的 Dateframe对象
sheet_dict['2015']

# 1.5查看2015 excel表中的 Dateframe对象 的基本信息
sheet_dict['2015'].info()

# 1.6查看2015 excel表中的 Dateframe对象 的基本统计信息
sheet_dict['2015'].describe()

# 1.7查看字典中每个Df对象（即每个sheet表）的基本信息和统计信息
# step1 遍历，获取到每个sheet表名
for i in sheet_names:
    print(i)           # i是每个sheet_name

# step2 打印每个sheet表的基本信息和统计信息
print(sheet_dict[i].info())
print(sheet_dict[i].describe())

sheet_dict

# 数据预处理

In [ ]:
# 需要处理的数据：1、删除缺失值。2.过滤出异常的数据. 3.固定时间节点，以每年最后一天作为当年的分析节点
    # 2.1 遍历，获取到每张表(2.1-2.4均为for循环内语句，需Tab缩进)
for i in sheet_names[:-1]:   #除开最后的一张表'会员等级'，取前面所有
    print(i)           # i是每个sheet_name

    # 2.2删除缺失值
    sheet_dict[i] = sheet_dict[i].dropna()
    sheet_dict[i].dropna(inplace=True)   # 直接改原表

    # 2.3过滤出不合理的数据(每个表的‘订单金额’列，取订单金额大于1的订单)
    sheet_dict[i] = sheet_dict[i][sheet_dict[i]['订单金额'] > 1 ]    # sheet_dict[i][Df对象['列名']] 或者 sheet_dict[i][Df对象.列名]
    #sheet_dict[i] = sheet_dict[i].query('"订单金额">1')  

    # 2.4固定时间节点，以每年的最后一天作为当前的分析节点
    sheet_dict[i]['max_year_date'] = sheet_dict[i]['提交日期'].max()  #生成一个新列'max_year_date'，取每张表内提交时间最大的那个值

# 2.5查看处理后数据
for i in sheet_names:
    # 打印每个sheet表的 基本信息 和 统计信息
    print(sheet_dict[i].info())
    print(sheet_dict[i].describe())
    print(i)

# 2.6把上述的四张表（对应的df对象）合并为一个df对象
sheet_dict['2015']                 # 单个Df对象
list(sheet_dict.values())[:-1]     # sheet_dict是一个字典，先将value值全取出来，用list做成一个列表（除开最后一张表）
pd.concat(list(sheet_dict.values())[:-1],ignore_index=True)    # 用concat将多个列表垂直合并，并忽视单个表的各自索引，重新生成索引
df_merge = pd.concat(list(sheet_dict.values())[:-1],ignore_index=True) 
df_merge

# 2.7为了好区分，给Df对象新增列 year
df_merge['year'] = df_merge['提交日期'].dt.year         # 如果“提交日期”列的日期数据类型为 datetime,则可以使用dt.year函数，提取日期

# 2.8给表新增一列，date_interval, 表示本订单购买时间据统计节点时间的差值
df_merge['date_interval'] = df_merge['max_year_date'] - df_merge['提交日期']

# 2.9把新增的列 date_interval 转为 int 类型 （现是日期数据）
df_merge['date_interval'] = df_merge['date_interval'].dt.days   # 同2.7
df_merge


# 数据的统计和分析

In [ ]:
# 3.1基于year 和会员ID分组，统计RFM三项基本数据
# R:recency(最近一次购买时间)，F:frequency(购买次数)，M:money(购买金额)
rfm_gb = df_merge.groupby(['year','会员ID'],as_index=False).agg({
   'date_interval':'min',
   '订单号':'count',
   '订单金额':'sum' 
})

# 3.2修改列名
rfm_gb.columns = ['year','会员ID','r','f','m']

# 3.3分别查看r,f,m三列值的分布情况
rfm_gb.iloc[:,2:].describe().T

# 3.4划分区间，分别给出RFM评分（R越小分越高，F越大分越高。M越大分越高）
#思路一，我们给定区间数，系统自动划分范围
pd.cut(rfm_gb['r'],bins=3)     #分箱，分成三个区间，每个区间包右不包左

#思路二，由我们手动指定区间范围，系统自动划分区间数
r_bins = [-1,51,187,362]             #由#3.3查看选取的每个列最小，最大值和25%，75%数等来选取
f_bins = [0,8,15,21]                 #区间范围包左不包右，所以第一位-1
m_bins = [200,3500,11111,100000]     #（-1,51],(51,187],(187,362]
pd.cut(rfm_gb['r'],bins=r_bins)
pd.cut(rfm_gb['f'],bins=f_bins)
pd.cut(rfm_gb['m'],bins=m_bins)

#思路三，基于我们手动指定区间范围，给出每个范围的评分（三分法，低中高）
rfm_gb['r_label'] = pd.cut(rfm_gb['r'],bins=r_bins,labels=[3,2,1])   # r：最近购买的时间，越小评分越高
rfm_gb['f_label'] = pd.cut(rfm_gb['f'],bins=f_bins,labels=[1,2,3])   # f：购买次数，越大评分越高
rfm_gb['m_label'] = pd.cut(rfm_gb['m'],bins=m_bins,labels=[1,2,3])   # m：购买金额，越大评分越高

#思路四，基于实际开发要求的分组区间数与范围，进行调整代码         range(1,4)包左不包右，值1 2 3 ，range(3,0,-1),值3 2 1（步长-1）
rfm_gb['r_label'] = pd.cut(rfm_gb['r'],bins=r_bins,labels=[i for i in range(len(r_bins)-1,0,-1)])   # r：最近购买的时间，越小评分越高
rfm_gb['f_label'] = pd.cut(rfm_gb['f'],bins=f_bins,labels=[i for i in range(1,len(f_bins))])        # f：购买次数，越大评分越高
rfm_gb['m_label'] = pd.cut(rfm_gb['m'],bins=m_bins,labels=[i for i in range(1,len(m_bins))])        # m：购买金额，越大评分越高

# 3.5统计每个会员的RFM评分 (采取方案：拼接)
#step1:转化类型，(r_label,f_label,m_label   从Categories分类类型转化为str字符串)
rfm_gb['r_label'] = rfm_gb['r_label'].astype(str)
rfm_gb['f_label'] = rfm_gb['f_label'].astype(str)
rfm_gb['m_label'] = rfm_gb['m_label'].astype(str)
##如果有无用列需要删除    rfm_gb.drop('列名'，axis=1,inplace=True)

#step2:拼接评分
rfm_gb['rfm_group'] = rfm_gb['r_label'] + rfm_gb['f_label'] + rfm_gb['m_label']
rfm_gb


# 导出结果

In [ ]:
# 4.1导出结果到Excel中,忽略索引
rfm_gb.to_excel('./结果数据.xlsx',index=False)

# 4.2.1导出结果到Mysql中
#step1创建引擎对象
engine = creat_engine('mysql+pymysql://root=123456@localhost:3306/rfm_gb?charset=utf8') 
# root（用户）123456是密码 ，localhost 主机地址 3306 mysql默认端口，rfm_gb是要导入mysql的库名已存在的数据库）

# 4.2.2具体的导出到mysql表中
# 参1：存储结构的数据表名，参2：引擎对象，参3：忽略索引，参4：如果表存在就替换数据，添加为append
rfm_gb.to_sql('rfm_table',engine,index=False,if_exists='replace')

# 4.2.3查看数据
pd.read_sql('select * from rfm_table',engine)

# 数据可视化

In [ ]:
# 5.1准备可视化的工具，即rfm_group（分组结果评分），year（统计年份），number（评分个数）
display_data = rfm_gb.groupby(['rfm_group','year'],as_index=False).agg({'会员ID':'count'})
display_data

# 5.2修改列名
display_data.columns = ['rfm_gb','year','number']
# 把number列的类型（字符串）转成int类型
display_data['number'] = display_data['number'].astype(int)

# 5.3绘制图形
# 颜色池
range_color = ['#313695','#4575b4','#74add1','#abd9e9','#e0f3f8','#fffbf'
               '#fee090','#fdae61','#f46d43','#d73027','#a50026']
range_max = int(display_data['number'].max())
c = (
    Bar3D() #设置一个3D柱形图对象
.add("", # 图例
     [d.tolist() for d in display_data.values], # 数据
     xaxis3d_opts=opts.Axis3DOpts(type_="category",name='分组名称'),       # x轴数据类型，名称，rfm_group
     yaxis3d_opts=opts.Axis3DOpts(type_="category",name='年份'),           # y轴数据类型，名称，year
     zaxis3d_opts=opts.Axis3DOpts(type_="value",name='会员数量'),        # z轴数据类型，名称，number
     )
     .set_global_opts( #全局设置
        visualmap_opts=opts.VisualMapOpts(max_=range_max,range_color=range_color), #设置颜色，极不同取值对应的颜色
         title_opts=opts.TitleOpts(title="RFM分组结果"), #设置标题
           )
)
c.render()  #数据保存到本地的网页中
# c.render_notebook() #在notebook中显示



## 结论分析

In [ ]:
# 1、重点人群分布在212这个分组
# 2、2015年的212这组会员人数最多，为41个，但从2015到2018起人数逐渐减少
# 3、总会员群体数量逐年减少
# 4、212、213、211人群占总数的大头，333组的人群无论是每年还是四年总和都很少
# 5、212和213：购买频率一般，购买单数较少但购买的金额可以，可以作为可发展的一般性群体
# 6、333组用户购买频率最少，但每次购买时订单数与金额都最大，所以可以考虑为其专门预留对应货源渠道和提供相关服务
